# Qwen2.5-3B Sentiment Pipeline (Gold Commodity Eval)

This notebook runs three stages:
1. **Zero-shot baseline** with `Qwen/Qwen2.5-3B-Instruct` on the Gold Commodity dataset.
2. **qLoRA fine-tuning** of `Qwen/Qwen2.5-3B-Instruct` on full FinancialPhraseBank, then evaluation on Gold Commodity.
3. **Random-search hyperparameter tuning** for Qwen qLoRA, then final re-evaluation on Gold Commodity.

All stage metrics are logged locally for comparison without W&B.

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [2]:
%%capture
!pip install -U unsloth transformers==4.56.2 trl==0.22.2 peft bitsandbytes accelerate datasets scikit-learn matplotlib pandas python-dotenv

## Device Check


In [3]:
import subprocess
import torch

print("=== nvidia-smi (if available) ===")
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout.strip():
        print(res.stdout)
    else:
        print(res.stderr.strip() or "nvidia-smi returned no output")
except FileNotFoundError:
    print("nvidia-smi not found on this machine.")

if torch.cuda.is_available():
    device_type = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"

print(f"Detected runtime device: {device_type}")
if device_type == "cuda":
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

=== nvidia-smi (if available) ===
Wed Apr  8 02:40:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.44.01              Driver Version: 590.44.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:01:00.0 Off |                  N/A |
| 36%   31C    P8             28W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------

## 1. Setup and Reproducibility

In [4]:
import os
import re
import gc
import math
import time
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def load_dotenv_walk(start: Path) -> bool:
    for parent in [start, *start.parents]:
        env_file = parent / '.env'
        if env_file.exists():
            load_dotenv(env_file, override=False)
            print(f'Loaded .env from: {parent}')
            return True
    return False

_ = load_dotenv_walk(Path(os.getcwd()))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

print(f'Device: {DEVICE}')
print(f'DType : {DTYPE}')

Device: cuda
DType : torch.bfloat16


## 2. W&B + Dataset Loading (Gold + FinancialPhraseBank)

In [5]:
from datasets import load_dataset, Dataset
import pandas as pd

# Load dataset
gold_ds = load_dataset("SaguaroCapital/sentiment-analysis-in-commodity-market-gold")
split_name = "test" if "test" in gold_ds else "train"
gold_raw_df = gold_ds[split_name].to_pandas()

# Text column
sentence_col = "News"

# One-hot label columns
label_cols = [
    "Price Direction Up",
    "Price Direction Constant",
    "Price Direction Down"
]

# Check expected columns exist
missing = [c for c in [sentence_col] + label_cols if c not in gold_raw_df.columns]
if missing:
    raise ValueError(
        f"Missing expected columns: {missing}. Available columns: {list(gold_raw_df.columns)}"
    )

# Convert one-hot labels into class names
def get_label(row):
    if row["Price Direction Up"] == 1:
        return "positive"
    elif row["Price Direction Constant"] == 1:
        return "neutral"
    elif row["Price Direction Down"] == 1:
        return "negative"
    else:
        return "unknown"

gold_raw_df["label_text"] = gold_raw_df.apply(get_label, axis=1)

# Final evaluation dataframe used by batched_generate_labels / evaluate_predictions
gold_eval_df = pd.DataFrame({
    "sentence": gold_raw_df[sentence_col].astype(str),
    "label_text": gold_raw_df["label_text"].astype(str)
})

# Optional: create a Hugging Face Dataset with a text column for trainer eval use
# This is only useful if you want to pass this dataset into SFTTrainer as eval_dataset
gold_raw_df["text"] = gold_raw_df[sentence_col].astype(str)
eval_ds = Dataset.from_pandas(gold_raw_df[["text"]], preserve_index=False)

print(f"Gold eval samples: {len(gold_eval_df)}")
print(gold_eval_df["label_text"].value_counts())
print(gold_eval_df.head())

print("\nTrainer eval dataset:")
print(eval_ds)
print(eval_ds.column_names)

/common/home/users/r/rohitht.2022/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gold eval samples: 2114
label_text
positive    881
negative    757
unknown     394
neutral      82
Name: count, dtype: int64
                                            sentence label_text
0  Gold / Silver / Copper futures - weekly outloo...    unknown
1  gold to be a safe-haven again; sell crude on r...    unknown
2  feb. gold settles at $1,097,90/oz on comex, do...   negative
3  dec gold rises 30c to $443.40/oz in morning ny...   positive
4    Gold holds modest losses after Chicago PMI miss   negative

Trainer eval dataset:
Dataset({
    features: ['text'],
    num_rows: 2114
})
['text']


In [6]:
# Download FinancialPhraseBank (75% agreement)
!curl -L "https://raw.githubusercontent.com/maxwellsarpong/NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt" -o Sentences_75Agree.txt

fpb_df = pd.read_csv(
    'Sentences_75Agree.txt',
    sep='@',
    header=None,
    names=['sentence', 'label_text'],
    engine='python',
    encoding='latin-1',
)
fpb_df['sentence'] = fpb_df['sentence'].str.strip()
fpb_df['label_text'] = fpb_df['label_text'].str.strip().str.lower()

print(f'FPB full samples: {len(fpb_df)}')
print(fpb_df['label_text'].value_counts())

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  452k  100  452k    0     0  1131k      0 --:--:-- --:--:-- --:--:-- 1128k
FPB full samples: 3453
label_text
neutral     2146
positive     887
negative     420
Name: count, dtype: int64


## 6. Random Search Tuning (Qwen qLoRA) + Final Evaluation on Gold

This stage uses a small random search over qLoRA hyperparameters with early stopping to keep tuning fast on GPU clusters.

In [7]:
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
)

LABELS = ['negative', 'neutral', 'positive']
LABEL_SET = set(LABELS)

def make_prompt(sentence: str) -> str:
    return (
        'You are a financial sentiment classifier.\n'
        'Return exactly one label: negative, neutral, or positive.\n\n'
        f'Sentence: {sentence}\n'
        'Label:'
    )

def parse_label(text: str) -> str:
    t = text.strip().lower()
    m = re.search(r'(negative|neutral|positive)', t)
    return m.group(1) if m else 'neutral'

@torch.inference_mode()
def batched_generate_labels(model, tokenizer, sentences, batch_size=32, max_new_tokens=3):
    preds = []
    tokenizer.padding_side = 'left'
    prompts = [make_prompt(s) for s in sentences]

    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i : i + batch_size]
        enc = tokenizer(
            chunk,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512,
        ).to(model.device)

        out = model.generate(
            **enc,
            do_sample=False,
            temperature=None,
            top_p=None,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

        gen_tokens = out[:, enc['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        preds.extend([parse_label(t) for t in texts])

    return preds

def evaluate_predictions(true_labels, pred_labels):
    true_labels = [x.strip().lower() for x in true_labels]
    pred_labels = [x.strip().lower() if x.strip().lower() in LABEL_SET else 'neutral' for x in pred_labels]

    metrics = {
        'macro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='macro', zero_division=0),
        'micro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='micro', zero_division=0),
        'weighted_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='weighted', zero_division=0),
        'f1_negative': f1_score(true_labels, pred_labels, labels=['negative'], average='macro', zero_division=0),
        'f1_neutral': f1_score(true_labels, pred_labels, labels=['neutral'], average='macro', zero_division=0),
        'f1_positive': f1_score(true_labels, pred_labels, labels=['positive'], average='macro', zero_division=0),
        'accuracy': accuracy_score(true_labels, pred_labels),
        'mcc': matthews_corrcoef(true_labels, pred_labels),
    }

    metrics['classification_report'] = classification_report(
        true_labels, pred_labels, labels=LABELS, zero_division=0
    )
    metrics['confusion_matrix'] = confusion_matrix(true_labels, pred_labels, labels=LABELS)
    return metrics

def print_metrics(title, metrics):
    print('=' * 80)
    print(title)
    print('=' * 80)
    print(metrics['classification_report'])
    print(f"Macro F1   : {metrics['macro_f1']:.4f}")
    print(f"Micro F1   : {metrics['micro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
    print(f"Accuracy   : {metrics['accuracy']:.4f}")
    print(f"MCC        : {metrics['mcc']:.4f}")
    print(f"F1 neg     : {metrics['f1_negative']:.4f}")
    print(f"F1 neu     : {metrics['f1_neutral']:.4f}")
    print(f"F1 pos     : {metrics['f1_positive']:.4f}")

def log_run_to_wandb(run_name, config_dict, metrics):
    # W&B logging has been disabled
    print(f"[W&B Disabled] Would have logged run: {run_name}")

## 4. Baseline: Qwen2.5-3B-Instruct (Non-Finetuned) on Gold Commodity

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, use_fast=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype=DTYPE,
    device_map='auto',
    load_in_4bit=True,
)
qwen_model.eval()

qwen_preds = batched_generate_labels(
    qwen_model,
    qwen_tokenizer,
    gold_eval_df['sentence'].tolist(),
    batch_size=64,
    max_new_tokens=3,
)

qwen_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), qwen_preds)
print_metrics('Qwen2.5-3B-Instruct (Zero-shot) on Gold Commodity', qwen_metrics)

log_run_to_wandb(
    run_name='qwen2.5-3b-instruct_zero_shot_gold',
    config_dict={
        'model_name': QWEN_MODEL_NAME,
        'stage': 'baseline_zero_shot',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        'quantization': '4bit',
        'batch_size': 64,
    },
    metrics=qwen_metrics,
)

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.91s/it]
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen2.5-3B-Instruct (Zero-shot) on Gold Commodity
              precision    recall  f1-score   support

    negative       0.75      0.64      0.69       757
     neutral       0.06      0.94      0.11        82
    positive       0.76      0.12      0.20       881

   micro avg       0.32      0.39      0.35      1720
   macro avg       0.53      0.57      0.34      1720
weighted avg       0.73      0.39      0.42      1720

Macro F1   : 0.3357
Micro F1   : 0.3479
Weighted F1: 0.4153
Accuracy   : 0.3155
MCC        : 0.2684
F1 neg     : 0.6938
F1 neu     : 0.1088
F1 pos     : 0.2045
[W&B Disabled] Would have logged run: qwen2.5-3b-instruct_zero_shot_gold


## 5. Fine-tune LLaMA3.2-3B-Instruct (Full FPB), Then Evaluate on Gold

In [9]:
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

LLAMA_MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'
LLAMA_ADAPTER_DIR = 'qlora_adapters/llama3.2-3b-fpb-full-default'
MAX_SEQ_LEN = 1024

def make_chat_train_text(tokenizer, sentence, label):
    messages = [
        {
            'role': 'system',
            'content': 'You are a financial sentiment classifier. Reply with one word: negative, neutral, or positive.',
        },
        {'role': 'user', 'content': f'Classify sentiment:\n{sentence}'},
        {'role': 'assistant', 'content': label},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

llama_model, llama_tokenizer = FastLanguageModel.from_pretrained(
    model_name=LLAMA_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

llama_model = FastLanguageModel.get_peft_model(
    llama_model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

train_ds = Dataset.from_dict({
    'text': [make_chat_train_text(llama_tokenizer, s, y) for s, y in zip(fpb_df['sentence'], fpb_df['label_text'])]
})

trainer = SFTTrainer(
    model=llama_model,
    tokenizer=llama_tokenizer,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    data_collator=DataCollatorForSeq2Seq(tokenizer=llama_tokenizer),
    packing=False,
    args=SFTConfig(
        output_dir='outputs/llama3.2-3b-fpb-full-default',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim='adamw_8bit',
        logging_steps=20,
        save_strategy='epoch',
        seed=SEED,
        report_to='none',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
    response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
)

train_out = trainer.train()
llama_model.save_pretrained(LLAMA_ADAPTER_DIR)
llama_tokenizer.save_pretrained(LLAMA_ADAPTER_DIR)

FastLanguageModel.for_inference(llama_model)
llama_model.eval()

llama_preds = batched_generate_labels(
    llama_model,
    llama_tokenizer,
    gold_eval_df['sentence'].tolist(),
    batch_size=64,
    max_new_tokens=3,
)

llama_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), llama_preds)
print_metrics('LLaMA3.2-3B qLoRA (Default Hyperparams) on Gold Commodity', llama_metrics)

log_run_to_wandb(
    run_name='llama3.2-3b-qlora_default_fullfpb_gold',
    config_dict={
        'model_name': LLAMA_MODEL_NAME,
        'adapter_dir': LLAMA_ADAPTER_DIR,
        'stage': 'finetune_default',
        'dataset_train': 'financialphrasebank_75agree_full',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'epochs': 3,
        'lr': 2e-4,
        'batch_size': 8,
        'grad_accum': 4,
        'train_loss': float(train_out.training_loss),
    },
    metrics=llama_metrics,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_3422071/488260591.py:2: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.561 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Filter (num_proc=64): 100%|██████████| 3453/3453 [00:03<00:00, 1026.17 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,453 | Num Epochs = 3 | Total steps = 324
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,0.239400
40,0.077400
60,0.084300
80,0.074800
100,0.068800
120,0.049100
140,0.030500
160,0.025500
180,0.040300
200,0.018600


LLaMA3.2-3B qLoRA (Default Hyperparams) on Gold Commodity
              precision    recall  f1-score   support

    negative       0.84      0.87      0.85       757
     neutral       0.12      0.70      0.20        82
    positive       0.86      0.82      0.84       881

   micro avg       0.68      0.83      0.75      1720
   macro avg       0.60      0.79      0.63      1720
weighted avg       0.81      0.83      0.81      1720

Macro F1   : 0.6299
Micro F1   : 0.7486
Weighted F1: 0.8137
Accuracy   : 0.6788
MCC        : 0.5658
F1 neg     : 0.8516
F1 neu     : 0.1996
F1 pos     : 0.8384
[W&B Disabled] Would have logged run: llama3.2-3b-qlora_default_fullfpb_gold


## 6. Random Search Tuning (Qwen qLoRA) + Final Evaluation on Gold

This stage uses a small random search over qLoRA hyperparameters with early stopping to keep tuning fast on GPU clusters.

In [18]:
import os
import gc
import random
import torch

RANDOM_SEARCH_TRIALS = int(os.getenv("RANDOM_SEARCH_TRIALS", 2))
RANDOM_SEARCH_SEED = int(os.getenv("RANDOM_SEARCH_SEED", SEED))
rng = random.Random(RANDOM_SEARCH_SEED)

fixed_params = {
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,
    "batch_size": 8,
    "grad_accum": 4,
    "epochs": 3,
}

if len(train_ds) == 0:
    raise ValueError("train_ds is empty")
if "text" not in train_ds.column_names:
    raise ValueError(f"train_ds missing 'text' column: {train_ds.column_names}")

print("train_ds length:", len(train_ds))
print("sample text:")
print(train_ds[0]["text"][:500])

model = None
tok = None
trainer = None

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

try:
    model, tok = FastLanguageModel.from_pretrained(
        model_name=QWEN_MODEL_NAME,
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=fixed_params["lora_r"],
        lora_alpha=fixed_params["lora_alpha"],
        lora_dropout=fixed_params["lora_dropout"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tok,
        train_dataset=train_ds,
        dataset_text_field="text",
        max_seq_length=1024,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tok),
        packing=False,
        args=SFTConfig(
            output_dir="outputs/qwen-default-matching-llama",
            num_train_epochs=fixed_params["epochs"],
            per_device_train_batch_size=fixed_params["batch_size"],
            gradient_accumulation_steps=fixed_params["grad_accum"],
            learning_rate=fixed_params["learning_rate"],
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            weight_decay=0.01,
            optim="adamw_8bit",
            logging_steps=20,
            save_strategy="epoch",
            seed=SEED,
            report_to="none",
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
        ),
    )

    train_out = trainer.train()

    del trainer
    trainer = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    FastLanguageModel.for_inference(model)
    model.eval()

    preds = batched_generate_labels(
        model,
        tok,
        gold_eval_df["sentence"].tolist(),
        batch_size=64,
        max_new_tokens=3,
    )

    metrics = evaluate_predictions(gold_eval_df["label_text"].tolist(), preds)
    print_metrics("Qwen qLoRA (Matched LLaMA Default Hyperparams) on Gold Commodity", metrics)

except Exception as e:
    print(f"Run failed: {type(e).__name__}: {e}")

finally:
    if trainer is not None:
        del trainer
    if model is not None:
        del model
    if tok is not None:
        del tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

train_ds length: 3453
sample text:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 08 Apr 2026

You are a financial sentiment classifier. Reply with one word: negative, neutral, or positive.<|eot_id|><|start_header_id|>user<|end_header_id|>

Classify sentiment:
According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .<|eot_id|><|start_header_id|>assistant<|end_header_id|>

neutral<|eot_id|>
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.561 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/

Unsloth: Tokenizing ["text"] (num_proc=64): 100%|██████████| 3453/3453 [00:17<00:00, 201.85 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,453 | Num Epochs = 3 | Total steps = 324
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,1.920700
40,0.584800
60,0.547100
80,0.544100
100,0.529500
120,0.494600
140,0.490800
160,0.482800
180,0.473000
200,0.477600


Qwen qLoRA (Matched LLaMA Default Hyperparams) on Gold Commodity
              precision    recall  f1-score   support

    negative       0.86      0.87      0.86       757
     neutral       0.11      0.73      0.19        82
    positive       0.88      0.78      0.83       881

   micro avg       0.67      0.82      0.73      1720
   macro avg       0.61      0.79      0.63      1720
weighted avg       0.83      0.82      0.81      1720

Macro F1   : 0.6260
Micro F1   : 0.7345
Weighted F1: 0.8131
Accuracy   : 0.6660
MCC        : 0.5617
F1 neg     : 0.8641
F1 neu     : 0.1863
F1 pos     : 0.8277


In [20]:
# Retrain one final model on full FPB with fixed params, then evaluate/log
best = fixed_params
BEST_ADAPTER_DIR = 'qlora_adapters/qwen2.5-3b-fpb-fixed-params-best'

best_model, best_tok = FastLanguageModel.from_pretrained(
    model_name=QWEN_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

best_model = FastLanguageModel.get_peft_model(
    best_model,
    r=best['lora_r'],
    lora_alpha=best['lora_alpha'],
    lora_dropout=best['lora_dropout'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

best_trainer = SFTTrainer(
    model=best_model,
    tokenizer=best_tok,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    data_collator=DataCollatorForSeq2Seq(tokenizer=best_tok),
    packing=False,
    args=SFTConfig(
        output_dir='outputs/qwen2.5-3b-fpb-fixed-params-best',
        num_train_epochs=best['epochs'],
        per_device_train_batch_size=best['batch_size'],
        gradient_accumulation_steps=best['grad_accum'],
        learning_rate=best['learning_rate'],
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim='adamw_8bit',
        logging_steps=20,
        save_strategy='epoch',
        seed=SEED,
        report_to='none',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

print('Starting final training with fixed params...')
best_train_out = best_trainer.train()

best_model.save_pretrained(BEST_ADAPTER_DIR)
best_tok.save_pretrained(BEST_ADAPTER_DIR)

del best_trainer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

FastLanguageModel.for_inference(best_model)
best_model.eval()

best_preds = batched_generate_labels(
    best_model,
    best_tok,
    gold_eval_df['sentence'].tolist(),
    batch_size=4,
    max_new_tokens=2,
)

best_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), best_preds)
print_metrics('Qwen2.5-3B qLoRA (Fixed Params) on Gold Commodity', best_metrics)

log_run_to_wandb(
    run_name='qwen2.5-3b-qlora_fixed_params_fullfpb_gold',
    config_dict={
        'model_name': QWEN_MODEL_NAME,
        'adapter_dir': BEST_ADAPTER_DIR,
        'stage': 'finetune_fixed_params',
        'dataset_train': 'financialphrasebank_75agree_full',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        **best,
        'train_loss': float(best_train_out.training_loss),
    },
    metrics=best_metrics,
)

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.561 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Tokenizing ["text"] (num_proc=64): 100%|██████████| 3453/3453 [00:17<00:00, 200.61 examples/s]


Starting final training with fixed params...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,453 | Num Epochs = 3 | Total steps = 324
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,1.921000
40,0.584100
60,0.547100
80,0.544000
100,0.529200
120,0.494300
140,0.490500
160,0.482300
180,0.472700
200,0.477200


Qwen2.5-3B qLoRA (Fixed Params) on Gold Commodity
              precision    recall  f1-score   support

    negative       0.85      0.89      0.87       757
     neutral       0.10      0.74      0.18        82
    positive       0.90      0.74      0.81       881

   micro avg       0.65      0.80      0.72      1720
   macro avg       0.62      0.79      0.62      1720
weighted avg       0.84      0.80      0.80      1720

Macro F1   : 0.6180
Micro F1   : 0.7199
Weighted F1: 0.8042
Accuracy   : 0.6528
MCC        : 0.5519
F1 neg     : 0.8664
F1 neu     : 0.1786
F1 pos     : 0.8090
[W&B Disabled] Would have logged run: qwen2.5-3b-qlora_fixed_params_fullfpb_gold


## 7. Compact Comparison Table

In [21]:
comparison_df = pd.DataFrame([
    {'model': 'Qwen2.5-3B-Instruct (zero-shot)', **{k: v for k, v in qwen_metrics.items() if isinstance(v, (int, float, np.floating))}},
    {'model': 'Qwen2.5-3B qLoRA (default)', **{k: v for k, v in llama_metrics.items() if isinstance(v, (int, float, np.floating))}},
    {'model': 'Qwen2.5-3B qLoRA (random search best)', **{k: v for k, v in best_metrics.items() if isinstance(v, (int, float, np.floating))}},
])

display_cols = ['model', 'macro_f1', 'micro_f1', 'weighted_f1', 'accuracy', 'mcc', 'f1_negative', 'f1_neutral', 'f1_positive']
display(comparison_df[display_cols].sort_values('macro_f1', ascending=False).reset_index(drop=True))

,model,macro_f1,micro_f1,weighted_f1,accuracy,mcc,f1_negative,f1_neutral,f1_positive
0,Qwen2.5-3B qLoRA (default),0.629870,0.748565,0.813738,0.678808,0.565767,0.851588,0.199650,0.838372
1,Qwen2.5-3B qLoRA (random search best),0.617993,0.719875,0.804189,0.652791,0.551927,0.866365,0.178624,0.808989
2,Qwen2.5-3B-Instruct (zero-shot),0.335690,0.347939,0.415292,0.315516,0.268387,0.693790,0.108757,0.204523


## 8. Placeholder Save of Best Model


In [ ]:
from pathlib import Path

EXPORT_DIR = Path("experiments/sentiment_agent/saved_models/random_search_best_placeholder")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if "best_model" in globals() and "best_tok" in globals():
    best_model.save_pretrained(EXPORT_DIR)
    best_tok.save_pretrained(EXPORT_DIR)
    print(f"Saved best model/tokenizer to: {EXPORT_DIR}")
elif "llama_model" in globals() and "llama_tokenizer" in globals():
    llama_model.save_pretrained(EXPORT_DIR)
    llama_tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved fallback llama model/tokenizer to: {EXPORT_DIR}")
elif "baseline_model" in globals() and "baseline_tokenizer" in globals():
    baseline_model.save_pretrained(EXPORT_DIR)
    baseline_tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved baseline model/tokenizer to: {EXPORT_DIR}")
elif "model" in globals() and "tokenizer" in globals():
    model.save_pretrained(EXPORT_DIR)
    tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved generic model/tokenizer to: {EXPORT_DIR}")
else:
    placeholder = EXPORT_DIR / "README_PLACEHOLDER.txt"
    placeholder.write_text(
        "No in-memory model object found. This directory is reserved for the best model export.",
        encoding="utf-8",
    )
    print(f"Created placeholder file: {placeholder}")

Saved best model/tokenizer to: experiments/sentiment_agent/saved_models/random_search_best_placeholder


## 8. Cleanup (Optional)

In [ ]:
for var_name in ['qwen_model', 'qwen_tokenizer', 'llama_model', 'llama_tokenizer', 'best_model', 'best_tok', 'trainer', 'best_trainer']:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleanup complete.')

Cleanup complete.
